# Plugin Metadata

> Data structures for plugin metadata

In [ ]:
#| default_exp core.metadata

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

## CapabilityMeta

The `CapabilityMeta` dataclass stores metadata about a plugin, including its name, version, and runtime state.

In [ ]:
#| export
@dataclass
class ResourceRequirements:
    """Binary hard-facts about what a plugin needs to run (Phase 5a).
    
    Quantitative resource amounts (min_vram_mb, etc.) deliberately omitted
    per CR-7's reactive resource management reframing — plugin authors can't
    reliably estimate model × dtype × quantization combinatorics, and Blender-
    style variable-render capabilities can't estimate at all. The substrate uses
    these binary hard-facts purely for discovery filtering; actual resource
    contention is handled reactively by CR-7's eviction + retry flow.
    
    - `requires_gpu`: True if the plugin needs any GPU; the substrate gates
      execution on a system monitor reporting one is present.
    - `platforms`: e.g., ["linux-x64", "darwin-arm64"]. Empty list means no
      platform constraint declared (assume universal compatibility).
    - `accelerators`: e.g., ["cuda", "mps", "cpu"]. Informational; substrate
      doesn't auto-select but consumers can filter on the values.
    """
    requires_gpu: bool = False
    platforms: List[str] = field(default_factory=list)
    accelerators: List[str] = field(default_factory=list)

In [ ]:
#| export
@dataclass
class CapabilityMeta:
    """Metadata about a plugin."""
    name:str # Plugin's unique identifier
    version:str # Plugin's version string
    description:str="" # Brief description of the plugin's functionality
    # SG-35: `author` and `package_name` removed — author lives in pyproject.toml
    # + importlib.metadata; package_name is derivable from the import system.
    # `description` is retained and validated by SG-6's manifest checker.
    # Phase 5a: binary hard-facts for discovery filtering (no quantitative amounts
    # per CR-7 reactive reframing). Optional during the cascade window; None for
    # capabilities/legacy manifests that declare no resource constraints.
    resources:Optional["ResourceRequirements"]=None
    config_schema:Optional[Dict[str, Any]]=None # JSON Schema for plugin configuration
    instance:Optional[Any]=None # Plugin instance (PluginInterface subclass)
    enabled:bool=True # Whether the plugin is enabled
    last_executed:float=0.0 # Unix timestamp
    # SG-9: drift detection — set by CapabilityManager.load_capability when the live
    # worker's /config_schema disagrees with the manifest's stored config_schema.
    # `live_config_schema` holds the worker-reported shape so callers can pick
    # which to honor (substrate keeps using `config_schema` for defaults +
    # validation; tooling and the future plugin-config UI library can inspect
    # `live_config_schema` for the post-regenerate-manifest preview).
    config_schema_drift:bool=False
    live_config_schema:Optional[Dict[str, Any]]=None
    # Pass-2 Thread 3 (stage 2): set when the worker's live-derived
    # structural surface disagrees with the manifest's witness hash
    # (same CR-8 idiom as config_schema_drift; stage-4 adapter
    # compatibility matches against the RECORDED surface).
    structural_surface_drift:bool=False

## CapabilityInstance (CR-10)

CR-10 introduces multi-instance plugin support: a single discovered plugin can have multiple
loaded instances differing by configuration. `CapabilityInstance` is the per-instance runtime
state, keyed by `instance_id` in `CapabilityManager._instances` / `CapabilityManager.instances`.

The `CapabilityMeta` continues to hold per-plugin (per-name) discovery + canonical-instance state;
`CapabilityInstance` covers what differs across multiple loads of the same plugin: the proxy, the
config that initialized it, the enabled flag, last-executed timestamp, and creation time.

**Naming conventions** (constrained-pattern, per CR-10 Q-resolution):

- Default: `instance_id == capability_name` — backward-compat for single-instance code paths.
- Named: caller passes a constrained string (`{alphanumeric, _, -}+`, len ≤ 64).
- Auto-generated: caller passes `new_instance=True` and the substrate produces
  `{capability_name}-{6-char hex}`.

The first instance loaded for a plugin (default `instance_id == capability_name`) is the
"canonical" instance — code reading `CapabilityMeta.instance` continues to see it. Multi-instance
loads after the first don't disturb that canonical reference.

In [ ]:
#| export
@dataclass
class CapabilityInstance:
    """Per-instance runtime state for a loaded plugin (CR-10 multi-instance).
    
    Differs from CapabilityMeta in scope:
    - CapabilityMeta is per-plugin-name discovery + canonical-instance state.
    - CapabilityInstance is per-load-call runtime state.
    
    A plugin loaded with no instance_id (default) gets `instance_id == capability_name`
    and is the canonical instance referenced by CapabilityMeta.instance. Multi-instance
    loads (instance_id != capability_name) add entries to CapabilityManager.instances
    without changing the canonical reference.
    """
    instance_id: str  # Unique key in CapabilityManager.instances; default = capability_name
    capability_name: str  # The underlying discovered plugin's name (CapabilityMeta.name)
    config: Dict[str, Any] = field(default_factory=dict)  # Effective config used at initialize()
    # The actual proxy (RemoteCapabilityProxy) bound to this instance. Typed as Any
    # to avoid importing proxy.py here (proxy depends on interface; interface +
    # metadata stay decoupled per the dependency hierarchy).
    proxy: Optional[Any] = None
    enabled: bool = True  # Per-instance enable flag; substrate's execute_capability checks this
    last_executed: float = 0.0  # Unix timestamp of the most recent execute on this instance
    # Timezone-aware datetime — datetime.utcnow() is deprecated in Python 3.12+
    # per the CR-5 follow-up. The factory uses datetime.now(timezone.utc) at
    # instance-creation time.
    created_at: datetime = field(default_factory=lambda: datetime.now(timezone.utc))
    # CR-7: empirical resource tracking key. Populated by load_capability via
    # `compute_config_hash(config)` so per-execute sample recording can index
    # the EmpiricalResourceStore by (instance_id, config_hash). Two distinct
    # configs for the same instance get two distinct empirical records.
    # Defaults to empty string for back-compat with hand-constructed PluginInstances
    # in tests that don't go through load_capability.
    config_hash: str = ""
    # CR-7 / SG-33: per-instance concurrency cap for async execute. None means
    # unbounded (preserves pre-SG-33 behavior). When set, CapabilityManager creates
    # a lazy asyncio.Semaphore(max_concurrent_requests) keyed by instance_id and
    # gates execute_capability_async behind it. Sync execute_capability is NOT gated —
    # the cap is async-path only since sync callers can't await a semaphore.
    max_concurrent_requests: Optional[int] = None

In [ ]:
#| hide
# CR-10 regression: CapabilityInstance basics + datetime-aware created_at.
import time as _time_cr10

inst = CapabilityInstance(instance_id="whisper", capability_name="whisper")
# Defaults
assert inst.instance_id == "whisper"
assert inst.capability_name == "whisper"
assert inst.config == {}
assert inst.proxy is None
assert inst.enabled is True
assert inst.last_executed == 0.0
# created_at is timezone-aware (CR-5 follow-up discipline)
assert inst.created_at.tzinfo is not None, "created_at must be timezone-aware"

# Multi-instance: different instance_id, same capability_name
inst_a = CapabilityInstance(instance_id="whisper-base", capability_name="whisper",
                        config={"model": "base"})
inst_b = CapabilityInstance(instance_id="whisper-large", capability_name="whisper",
                        config={"model": "large"})
assert inst_a.instance_id != inst_b.instance_id
assert inst_a.capability_name == inst_b.capability_name == "whisper"
assert inst_a.config != inst_b.config

# Distinct created_at — at least monotonically non-decreasing within a wall-clock
# tick. We don't assert strict less-than because microsecond resolution may
# cluster two field-factory calls within the same instant.
assert inst_a.created_at <= inst_b.created_at

print("✓ CapabilityInstance dataclass: defaults + tz-aware created_at + multi-instance differentiation")

In [ ]:
#| export
@dataclass
class CapabilityLoadSpec:
    """One entry in `CapabilityManager.load_capabilities_concurrent`'s batch input (CR-10).
    
    Mirrors the positional arguments of `load_capability` so the concurrent helper
    can fan out load calls without repeating the per-spec instance_id /
    new_instance plumbing.
    
    - `meta`: the discovered CapabilityMeta to load (must have a `.manifest` attached).
    - `config`: initial configuration; falls through to persisted-or-schema-defaults
      when None (default-instance only; multi-instance starts fresh).
    - `instance_id`: explicit instance_id (validated against [A-Za-z0-9_-]{1,64}).
      None defaults to capability_name (single-instance backward compat).
    - `new_instance`: when True with instance_id=None, auto-generate
      `{capability_name}-{6-hex}`.
    """
    meta: Any  # CapabilityMeta — typed as Any to avoid forward-reference quirk under nbdev's late binding
    config: Optional[Dict[str, Any]] = None
    instance_id: Optional[str] = None
    new_instance: bool = False

### Example: Creating Plugin Metadata

In [ ]:
# Create plugin metadata
meta = CapabilityMeta(
    name="example_plugin",
    version="1.0.0",
    description="An example plugin",
    config_schema={
        "type": "object",
        "properties": {
            "model": {"type": "string", "default": "base"},
            "device": {"type": "string", "enum": ["cpu", "cuda"]}
        }
    }
)

print("CapabilityMeta instance:")
print(meta)
print(f"\nName: {meta.name}")
print(f"Version: {meta.version}")
print(f"Config Schema: {meta.config_schema}")
print(f"Enabled: {meta.enabled}")
print(f"Instance: {meta.instance}")

In [ ]:
# Test with minimal arguments
minimal_meta = CapabilityMeta(name="minimal", version="0.1.0")
print(f"Minimal CapabilityMeta: {minimal_meta}")

# Test equality
meta_copy = CapabilityMeta(name="minimal", version="0.1.0")
print(f"\nEquality test: {minimal_meta == meta_copy}")

Minimal PluginMeta: PluginMeta(name='minimal', version='0.1.0', description='', author='', package_name='', category='', interface='', config_schema=None, instance=None, enabled=True, last_executed=0.0)

Equality test: True


In [ ]:
# Phase 5a: ResourceRequirements round-trips cleanly, and CapabilityMeta
# accepts it as an optional field (None for capabilities declaring no constraints).
res = ResourceRequirements(
    requires_gpu=True,
    platforms=["linux-x64", "darwin-arm64"],
    accelerators=["cuda", "mps"],
)
assert res.requires_gpu is True
assert "linux-x64" in res.platforms

# ResourceRequirements defaults: no GPU, empty platforms (universal), empty accelerators.
default_res = ResourceRequirements()
assert default_res.requires_gpu is False
assert default_res.platforms == []
assert default_res.accelerators == []

# CapabilityMeta accepts resources; defaults to None when unconstrained.
typed_meta = CapabilityMeta(
    name="whisper-local",
    version="1.0.0",
    description="Local Whisper-based STT",
    resources=res,
)
assert typed_meta.resources.requires_gpu is True

legacy_meta = CapabilityMeta(name="legacy", version="0.0.1")
assert legacy_meta.resources is None

print("✓ ResourceRequirements + CapabilityMeta integration")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()